# Delft: a first run

This example downloads a ready case, the morning of Delft, runs it with one command and reads what the run wrote:
who took which mode and how long it took, how the iterations settled, the traffic on the links for cars, bikes and
walkers, one traveller's journey leg by leg, and the two pages a run can write, the map and the summary.

It needs `pip install openmobisim pyarrow`. The package itself has no Python dependency; `pyarrow` reads the Parquet
tables here. Everything below was run as shown, on a laptop (Apple M2 Pro, 12 cores).

## The case

The case is one of five published as a data record ([doi:10.5281/zenodo.24585967](https://doi.org/10.5281/zenodo.24585967)).
Its network and timetable are real; its travellers are synthetic, not observed and not calibrated.

In [1]:
import os, urllib.request, zipfile

urllib.request.urlretrieve("https://zenodo.org/records/24585967/files/delft.zip", "delft.zip")
zipfile.ZipFile("delft.zip").extractall()
sorted(os.listdir("delft"))

['LICENCE.md',
 'gtfs.zip',
 'links.parquet',
 'nodes.parquet',
 'run.toml',
 'trips.parquet']

`nodes.parquet` and `links.parquet` are the network, `gtfs.zip` the timetable, `trips.parquet` the travellers and
`run.toml` the settings. A link is for one mode: a street that cars, bikes and walkers all use is three links.

In [2]:
import collections
import pyarrow.parquet as pq

links = pq.read_table("delft/links.parquet").to_pylist()
for mode in ("car", "bike", "walk"):
    of_mode = [l for l in links if l["modes"] == mode]
    print(f"{mode:5} {len(of_mode):6} links, {sum(l['length_m'] for l in of_mode) / 1000:6.0f} km")

car    12460 links,    912 km
bike   20652 links,   1240 km
walk   55400 links,   2253 km


Each traveller is one row: the two ends of the trip, the second of departure (seconds after midnight), a class and
the modes open to that traveller. Here are the classes and what each may use.

In [3]:
travellers = pq.read_table("delft/trips.parquet").to_pylist()
print(travellers[0])
for (class_id, modes), count in sorted(collections.Counter((t["class_id"], t["modes"]) for t in travellers).items()):
    print(f"{class_id:22} {modes:24} {count:6}")

{'trip_id': 'od0000001', 'origin_lon_deg': 4.3908742, 'origin_lat_deg': 51.9660036, 'destination_lon_deg': 4.4039418, 'destination_lat_deg': 51.9684009, 'departure_s': 30279, 'class_id': 'car_and_bike', 'modes': 'car;bike;walk;transit'}
bike_only              bike;walk;transit          2663
car_and_bike           car;bike;walk;transit      5261
car_only               car;walk;transit           1264
from_outside_car       car                       16544
from_outside_transit   transit                    6325
from_outside_transit   transit;bike                 89
goods                  car                        1505
no_vehicle             walk;transit               3998


## The run

`run_file` reads the settings file `run.toml` in the folder, builds the network, the timetable and the travellers
from the files it names, and runs. The settings give the window, 07:30 to 09:30 (27 000 s to 34 200 s), and every
parameter at its value ([the parameters table](../parameters.md)).

In [4]:
import time
import openmobisim as oms

start = time.perf_counter()
run = oms.run_file("delft", out="delft-run")
print(f"run time {time.perf_counter() - start:.1f} s; ended by the stopping rule: {run.stopped_by_rule}")
sorted(os.listdir("delft-run"))

run time 3.9 s; ended by the stopping rule: True


['iterations.parquet',
 'links.parquet',
 'routes.parquet',
 'run_record.toml',
 'trips.parquet']

The run wrote four tables and its record. `run.trips`, `run.links`, `run.iterations`, `run.routes` and `run.record`
are their paths.

## Who took what, and how long it took

`trips.parquet` has one row per traveller: the mode taken and the travel time, door to door, as realised in the
last loading (the walk to and from the network or a stop included).

In [5]:
def by_mode(trips_file):
    """Per mode: the travellers who took it, their share (%) and their mean door-to-door time (min)."""
    trips = pq.read_table(trips_file).to_pylist()
    rows = {}
    for mode in ("car", "bike", "walk", "transit"):
        times = [t["travel_time_s"] for t in trips if t["mode"] == mode and t["finished"]]
        count = sum(t["mode"] == mode for t in trips)
        rows[mode] = (count, 100 * count / len(trips), sum(times) / len(times) / 60)
    return rows

print("mode     travellers  share (%)  mean door-to-door time (min)")
for mode, (count, share, minutes) in by_mode(run.trips).items():
    print(f"{mode:8} {count:10} {share:10.1f} {minutes:10.1f}")
trips = pq.read_table(run.trips).to_pylist()
print("unfinished at 09:30:", sum(not t["finished"] for t in trips))

mode     travellers  share (%)  mean door-to-door time (min)
car           22475       59.7        8.3
bike           4364       11.6       12.8
walk           1614        4.3       24.8
transit        9196       24.4       28.3
unfinished at 09:30: 0


A trip still under way when the window ends is unfinished and has no travel time; means are over finished trips, so
read them with the unfinished count. Here every traveller arrives.

## How the iterations settled

Iteration 0 runs on free-flow times and the timetable: everyone chooses as if the roads were empty, so the roads
overload and many trips do not finish. From then on, each iteration searches for better alternatives at the last
loading's times, lets a share of the travellers choose again, loads, and measures. `iterations.parquet` has one
row per iteration.

In [6]:
print("iteration  mean trip (min)  unfinished  offered a new choice (%)  switched (%)  would change if asked (%)")
for it in pq.read_table(run.iterations).to_pylist():
    print(f"{it['iteration']:9} {it['mean_trip_time_s'] / 60:16.2f} {it['unfinished']:11} {100 * it['offered_share']:25.1f}"
          f" {100 * it['switch_share']:13.2f} {100 * it['would_change_share']:26.2f}")

iteration  mean trip (min)  unfinished  offered a new choice (%)  switched (%)  would change if asked (%)
        0            21.37        4689                     100.0          0.00                       0.83
        1            15.99           0                      59.2         18.76                      14.33
        2            15.10           0                      43.5          8.75                      13.69
        3            14.64           0                      35.4          5.74                      11.68
        4            14.48           0                      30.2          3.78                       9.30
        5            14.43           0                      26.3          2.71                       7.65
        6            14.44           0                      23.2          1.94                       6.50
        7            14.41           0                      21.3          1.47                       5.60
        8            14.42           0        

The run stops when the mean trip time moved at most 0.3 % over the last five iterations and at most 0.5 % of the
trips found a cheaper alternative, or at the cap on iterations (`iteration_cap`). The last column is reported and
not judged: under the logit, the share of travellers who would hold another alternative if all were asked again.

## The traffic on the links

`links.parquet` has one row per link and time bin (300 s here) that someone entered: the cars with the time they
spent on the link, the bikes and the walkers. Joined with the network's lengths it gives the distance travelled on
the network per mode.

In [7]:
length_m = {l["link_id"]: l["length_m"] for l in links}
free_flow_s = {l["link_id"]: l["length_m"] / (l["car_speed_km_h"] / 3.6) for l in links if l["modes"] == "car"}
bins = pq.read_table(run.links).to_pylist()
for mode, column in (("car", "cars"), ("bike", "bikes"), ("walk", "walkers")):
    print(f"{mode:5} {sum(b[column] * length_m[b['link_id']] for b in bins) / 1000:9.0f} km on the network")

car      187209 km on the network
bike      12612 km on the network
walk       3069 km on the network


Cars queue; bikes and walkers travel at free flow. A link's free-flow time is its length divided by its free-flow
speed, so the time the cars lost on a link is the time they spent there minus the cars times the free-flow time.

In [8]:
spent = collections.Counter()
lost = collections.Counter()
for b in bins:
    if b["cars"]:
        spent[b["link_id"]] += b["car_time_s"]
        lost[b["link_id"]] += b["car_time_s"] - b["cars"] * free_flow_s[b["link_id"]]
print(f"car delay: {sum(lost.values()) / 3600:.0f} vehicle hours, {100 * sum(lost.values()) / sum(spent.values()):.1f} % of the cars' time on links")
road_class = {l["link_id"]: l["road_class"] for l in links}
print("the five links where the cars lost most time:")
for link_id, seconds in lost.most_common(5):
    print(f"  {road_class[link_id]:14} {length_m[link_id]:6.0f} m  {seconds / 3600:5.1f} vehicle hours")

car delay: 394 vehicle hours, 13.8 % of the cars' time on links
the five links where the cars lost most time:
  motorway_link     205 m   82.8 vehicle hours
  motorway_link      16 m   62.9 vehicle hours
  motorway_link     143 m   49.7 vehicle hours
  motorway_link     204 m   44.5 vehicle hours
  tertiary          233 m   37.4 vehicle hours


## One traveller, leg by leg

`routes.parquet` holds what each traveller took at the last iteration, one row per leg. A transit journey is a walk
to a stop, its rides (with the walks between them) and a walk to the door; a car, bike or walk trip is a walk to the
network, its route as a list of links, and a walk to the door. Here is the first transit traveller with two rides.

In [9]:
legs = collections.defaultdict(list)
for leg in pq.read_table(run.routes).to_pylist():
    legs[leg["trip_id"]].append(leg)
trip_id = next(t for t, its in legs.items() if sum(l["kind"] == "ride" for l in its) == 2)
trip = next(t for t in trips if t["trip_id"] == trip_id)
clock = lambda s: f"{int(s) // 3600:02}:{int(s) % 3600 // 60:02}:{int(s) % 60:02}"
print(f"{trip_id}: leaves the door at {clock(trip['departure_s'])}, arrives at {clock(trip['arrival_s'])}, {trip['travel_time_s'] / 60:.1f} min")
for leg in legs[trip_id]:
    if leg["kind"] == "ride":
        print(f"  ride           boards at stop {leg['board_stop']} at {clock(leg['board_s'])}, alights at stop {leg['alight_stop']} at {clock(leg['alight_s'])}")
    else:
        print(f"  {leg['kind']:14} {leg['seconds']:5.0f} s")

od0000016: leaves the door at 07:30:49, arrives at 08:29:42, 58.9 min
  walk_gap          67 s
  ride           boards at stop 3989648 at 07:42:48, alights at stop 3990650 at 07:57:00
  transfer_walk     15 s
  ride           boards at stop 3428643 at 08:00:00, alights at stop 3621208 at 08:14:00
  walk_gap         943 s


The traveller leaves the door at the departure second, walks to the stop and waits for the first run it can board;
that wait is part of the travel time.

## The map and the summary

A run writes two pages, each one self-contained HTML file that opens from disk with no server and no network.

In [10]:
run.map("delft-map.html")
run.summary("delft-summary.html")
for page in ("delft-map.html", "delft-summary.html"):
    print(f"{page}: {os.path.getsize(page) / 1e6:.1f} MB")

delft-map.html: 3.2 MB
delft-summary.html: 2.1 MB


![The map page of the run: Delft's roads, one ribbon per direction, coloured by volume over capacity](../_static/figures/delft_map.png)

The map page as it opens (the network is from OpenStreetMap, © OpenStreetMap contributors): road traffic by time
bin, with layers for transit lines, cyclists and walkers, the journeys between the busiest pairs of places, and, for
a link clicked, the pairs that use it. The summary page gives the run's numbers: the modes, when people left, the
roads, transit, bikes and walking, the iterations, the computation time and the record.

## The record, and running again

`run_record.toml` is the record of what was run. Its first part is the settings the run ran with, every parameter at
the value used; its second part is what the run found: how it ended and the fingerprints of its inputs.

In [11]:
record = open(run.record).read()
print(record[record.index("[result]\n"):record.index("[result.transit.gtfs_report]")].strip())

[result]
openmobisim_version = "0.2.0-dev"
window_end_used_s = 34200  # s: window_end_s, moved up to a whole link_bin when it was not one
iterations = 11
stopped_by_rule = true

[result.network]
source = "tables"
nodes = 21566
links = 88512
fingerprint = "3f87b0d8e1693b38"

[result.demand]
trips = 37649
fingerprint = "4ba8ec884eb6e048"

[result.transit]
stops = 1340
runs = 1707
fingerprint = "b78ffca832c14a3c"


The same inputs and settings give the same tables, on any machine and with any number of threads. `Run.open` opens
a finished run from its folder without running anything, for instance to write its pages later:

```python
run = oms.Run.open("delft-run")
```

To change a setting, edit `run.toml` (or copy the folder and edit the copy) and run again. The
[next example](amsterdam.ipynb) changes one from Python instead.